# Maternal Health Risk Prediction

## XGBoost Classifier Model

#### Import Required Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from xgboost import XGBClassifier
import pickle
import os

print("Starting XGBoost script with tuning...")

Starting XGBoost script with tuning...


#### Load Data

In [2]:
data = pd.read_csv("../Dataset/Maternal Health Risk Data Set.csv")
print("Data loaded. Shape:", data.shape)

data = data.copy()

Data loaded. Shape: (1014, 7)


#### Data Information

In [3]:
print("="*60)
print("Dataset Information")
print("="*60)

display(data.info())

print()

display(data.describe())

print()

display(data.describe(include="object"))

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1014 non-null   int64  
 1   SystolicBP   1014 non-null   int64  
 2   DiastolicBP  1014 non-null   int64  
 3   BS           1014 non-null   float64
 4   BodyTemp     1014 non-null   float64
 5   HeartRate    1014 non-null   int64  
 6   RiskLevel    1014 non-null   object 
dtypes: float64(2), int64(4), object(1)
memory usage: 55.6+ KB


None

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
count,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000
mean,29.871795,113.198225,76.460552,8.725986,98.665089,74.301775
std,13.474386,18.403913,13.885796,3.293532,1.371384,8.088702
min,10.000000,70.000000,49.000000,6.000000,98.000000,7.000000
25%,19.000000,100.000000,65.000000,6.900000,98.000000,70.000000
50%,26.000000,120.000000,80.000000,7.500000,98.000000,76.000000
75%,39.000000,120.000000,90.000000,8.000000,98.000000,80.000000
max,70.000000,160.000000,100.000000,19.000000,103.000000,90.000000


,RiskLevel
count,1014
unique,3
top,low risk
freq,406


#### Feature Engineering

In [4]:
if {"SystolicBP", "DiastolicBP"}.issubset(data.columns):
    data["PulsePressure"] = data["SystolicBP"] - data["DiastolicBP"]
    eps = 1e-3
    data["BPRatio"] = data["SystolicBP"] / (data["DiastolicBP"] + eps)

if {"BS", "HeartRate"}.issubset(data.columns):
    data["BS_HR_Interaction"] = data["BS"] * data["HeartRate"]

if {"Age", "SystolicBP"}.issubset(data.columns):
    data["Age_SystolicBP"] = data["Age"] * data["SystolicBP"]

TARGET_COL = "RiskLevel"

X = data.drop(columns=[TARGET_COL])
y = data[TARGET_COL]

# 🔥 FIX: Encode labels for XGBoost
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

print("Encoded classes:", list(le.classes_))
print("Features shape:", X.shape)

Encoded classes: ['high risk', 'low risk', 'mid risk']
Features shape: (1014, 10)


#### Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train set:", X_train.shape, "Test set:", X_test.shape)

Train set: (811, 10) Test set: (203, 10)


#### Pipeline (Scaler + XGBoost)

In [6]:
xgb_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("xgb", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        use_label_encoder=False
    ))
])

#### Hyperparameter Grid

In [7]:
param_grid = {
    "xgb__n_estimators": [100, 200, 300],
    "xgb__learning_rate": [0.01, 0.05, 0.1],
    "xgb__max_depth": [3, 5, 7],
    "xgb__subsample": [0.6, 0.8, 1.0],
    "xgb__colsample_bytree": [0.6, 0.8, 1.0],
    "xgb__gamma": [0, 1, 5],
    "xgb__reg_lambda": [1, 5, 10]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#### Grid Search

In [8]:
print("Running GridSearchCV...")

grid_search = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best CV Accuracy:", grid_search.best_score_)
print("Best Params:", grid_search.best_params_)

Running GridSearchCV...
Fitting 5 folds for each of 2187 candidates, totalling 10935 fits


C:\Users\viqua\anaconda3\envs\pilot\lib\site-packages\xgboost\training.py:200: UserWarning: [16:16:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best CV Accuracy: 0.8273801408770733
Best Params: {'xgb__colsample_bytree': 0.8, 'xgb__gamma': 0, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 7, 'xgb__n_estimators': 200, 'xgb__reg_lambda': 1, 'xgb__subsample': 1.0}


#### Evaluate Best XGBoost Model

In [9]:
best_model = grid_search.best_estimator_

print("Training best model on full training data...")
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("\n===== FINAL TEST PERFORMANCE (BEST XGBOOST) =====")
print("Test Accuracy:", acc)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

print("Script finished.")

Training best model on full training data...


C:\Users\viqua\anaconda3\envs\pilot\lib\site-packages\xgboost\training.py:200: UserWarning: [16:16:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== FINAL TEST PERFORMANCE (BEST XGBOOST) =====
Test Accuracy: 0.8620689655172413

Confusion Matrix:
 [[50  1  4]
 [ 2 66 13]
 [ 2  6 59]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.91      0.92        55
           1       0.90      0.81      0.86        81
           2       0.78      0.88      0.83        67

    accuracy                           0.86       203
   macro avg       0.87      0.87      0.87       203
weighted avg       0.87      0.86      0.86       203

Script finished.


#### Save XGBoost Model

In [11]:
print("="*70)
print("Saving XG Boost Model")
print("="*70)

os.makedirs("../Model", exist_ok=True)

MODEL_PATH = "../Model/xgboost.pkl"

with open(MODEL_PATH, "wb") as file:
    pickle.dump(best_model, file)

print("Model Saved Successfully")
print("Location :", MODEL_PATH)

Saving XG Boost Model
Model Saved Successfully
Location : ../Model/xgboost.pkl
